# UD2/UD3 - Clasificación: Titanic

Cuaderno guía para EDA, preparación y modelado en el reto **Titanic**. Incluye celdas TODO comentadas para que el alumnado decida.

## Dependencias y configuración
- Descarga previa: `kaggle competitions download -c titanic`.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")
DATA_DIR = Path("data")
TITANIC_DIR = DATA_DIR / "titanic"

## Utilidades

In [ ]:
def missing_report(df, top=20):
    miss = df.isna().mean().sort_values(ascending=False)
    return miss[miss > 0].head(top)

def eval_classification(y_true, y_pred, y_proba=None, label="modelo"):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
        except ValueError:
            pass
    print(label, metrics)
    return metrics

## Carga de datos

In [ ]:
train_ti_path = TITANIC_DIR / "train.csv"
test_ti_path = TITANIC_DIR / "test.csv"

if not train_ti_path.exists():
    raise FileNotFoundError(f"No se encontró {train_ti_path}. Descarga con Kaggle API y descomprime.")

ti_train = pd.read_csv(train_ti_path)
ti_test = pd.read_csv(test_ti_path) if test_ti_path.exists() else None
ti_train.shape, ti_test.shape if ti_test is not None else None

## EDA inicial

In [ ]:
ti_train.head()

In [ ]:
ti_train.info()

In [ ]:
ti_missing = missing_report(ti_train); ti_missing

### Visuales rápidas

In [ ]:
# Balance de clases
sns.countplot(data=ti_train, x='Survived')
plt.title('Balance de clases')
plt.tight_layout()

In [ ]:
# TODO: Añade gráficos bivariantes (Survived vs Sex/Pclass/Age)
# sns.barplot(data=ti_train, x='Sex', y='Survived')

## Manejo de faltantes
- Ejemplo: eliminar `Cabin`, imputar numéricas con mediana y categóricas con moda.
- Celda TODO para imputación alternativa de Age por título.

In [ ]:
cols_drop_ti = ['Cabin']
ti_train_clean = ti_train.drop(columns=[c for c in cols_drop_ti if c in ti_train.columns])
if ti_test is not None:
    ti_test_clean = ti_test.drop(columns=[c for c in cols_drop_ti if c in ti_test.columns])

num_cols_ti = ti_train_clean.select_dtypes(include=[np.number]).columns.tolist()
num_cols_ti.remove('Survived')
cat_cols_ti = [c for c in ti_train_clean.columns if c not in num_cols_ti + ['Survived']]

ti_train_clean[num_cols_ti] = ti_train_clean[num_cols_ti].fillna(ti_train_clean[num_cols_ti].median())
ti_train_clean[cat_cols_ti] = ti_train_clean[cat_cols_ti].fillna(ti_train_clean[cat_cols_ti].mode().iloc[0])

In [ ]:
# TODO: Imputar Age por título extraído de Name
# ti_train_clean['Title'] = ti_train_clean['Name'].str.extract(' ([A-Za-z]+)\.')
# title_medians = ti_train_clean.groupby('Title')['Age'].transform('median')
# ti_train_clean['Age'] = ti_train_clean['Age'].fillna(title_medians)

## Split y pipelines

In [ ]:
X_ti = ti_train_clean.drop(columns=['Survived'])
y_ti = ti_train_clean['Survived']

X_train_ti, X_val_ti, y_train_ti, y_val_ti = train_test_split(
    X_ti, y_ti, test_size=0.2, random_state=42, stratify=y_ti
)

numeric_pipe_ti = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe_ti = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocess_ti = ColumnTransformer([
    ("num", numeric_pipe_ti, num_cols_ti),
    ("cat", categorical_pipe_ti, cat_cols_ti),
])

## Modelos base y métricas

In [ ]:
models_ti = {
    "LogReg": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(random_state=42, n_estimators=300),
}

results_ti = {}
for name, model in models_ti.items():
    pipe = Pipeline([("prep", preprocess_ti), ("model", model)])
    pipe.fit(X_train_ti, y_train_ti)
    pred = pipe.predict(X_val_ti)
    proba = pipe.predict_proba(X_val_ti)[:, 1] if hasattr(pipe, "predict_proba") else None
    results_ti[name] = eval_classification(y_val_ti, pred, proba, label=name)
results_ti

## Optimización de hiperparámetros (RandomForest)

In [ ]:
rf_ti = RandomForestClassifier(random_state=42)
param_grid_ti = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_leaf": [1, 2, 4],
}

rf_pipe_ti = Pipeline([("prep", preprocess_ti), ("model", rf_ti)])

grid_ti = GridSearchCV(rf_pipe_ti, param_grid_ti, cv=5, scoring="roc_auc", n_jobs=-1)
# grid_ti.fit(X_ti, y_ti)
# grid_ti.best_params_, grid_ti.best_score_

## Evaluación adicional (curva ROC, matriz de confusión)

In [ ]:
# TODO: Tras entrenar el mejor modelo, dibuja ROC y matriz de confusión
# from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay
# RocCurveDisplay.from_estimator(grid_ti.best_estimator_, X_val_ti, y_val_ti)
# ConfusionMatrixDisplay.from_estimator(grid_ti.best_estimator_, X_val_ti, y_val_ti, normalize='true')
# plt.show()

## Exportar limpio (opcional)

In [ ]:
# ti_train_clean.to_csv(DATA_DIR / 'titanic_train_clean.csv', index=False)

## Notas finales
- Flujo: EDA → limpieza → split → pipeline → modelos → tuning.
- Activa celdas TODO según decisiones (imputaciones, visuales, evaluación).